In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor, XGBClassifier
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import joblib

In [2]:
X_train = pd.read_csv('../data/processed/train_test_splits/X_train_sm_clean.csv')
X_test = pd.read_csv('../data/processed/train_test_splits/X_test_sm_clean.csv')
y_train = pd.read_csv('../data/processed/train_test_splits/y_train_sm_clean.csv').squeeze()
y_test = pd.read_csv('../data/processed/train_test_splits/y_test_sm_clean.csv').squeeze()


In [5]:

surv_model = XGBRegressor(
    n_estimators=1200,
    learning_rate=0.03,
    max_depth=7,
    min_child_weight=3,
    subsample=0.85,
    colsample_bytree=0.85,
    gamma=0,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=42
)
surv_model.fit(X_train, y_train)

r2_surv = surv_model.score(X_test, y_test)
mae_surv = mean_absolute_error(y_test, surv_model.predict(X_test))
rmse_surv = np.sqrt(mean_squared_error(y_test, surv_model.predict(X_test)))

print(f"\n🏆 SURVIVAL MODEL:")
print(f"   R²:   {r2_surv:.4f}")
print(f"   MAE:  {mae_surv:.2f} months")
print(f"   RMSE: {rmse_surv:.2f} months")

imp_surv = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': surv_model.feature_importances_
}).sort_values(by='Importance', ascending=False)
print("\n   TOP 10 FEATURES:")
for i, row in imp_surv.head(10).iterrows():
    print(f"   {row['Feature']}: {row['Importance']:.4f}")

if r2_surv > 0.70:
    print("   ⚠️ WARNING: R² > 0.70, check for remaining leaks!")
elif r2_surv > 0.35:
    print(f"   ✅ ACCEPTABLE! R²={r2_surv:.4f} is realistic.")
    joblib.dump(surv_model, '../models/survival_model_v1_clean.pkl')
else:
    print(f"   ⚠️ R²={r2_surv:.4f} is low, but might still be useful.")
    joblib.dump(surv_model, '../models/survival_model_v1_clean.pkl')



🏆 SURVIVAL MODEL:
   R²:   0.6660
   MAE:  22.58 months
   RMSE: 32.60 months

   TOP 10 FEATURES:
   market_saturation: 0.3118
   product_type_app: 0.2292
   has_founders: 0.0700
   category_encoded: 0.0363
   desc_5: 0.0174
   price_vs_category: 0.0103
   product_type_digital_app: 0.0097
   product_type_encoded: 0.0091
   info_completeness: 0.0085
   desc_56: 0.0075
   ✅ ACCEPTABLE! R²=0.6660 is realistic.


In [13]:
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score
import numpy as np
import joblib

# Load clean data
X_train = pd.read_csv('../data/processed/train_test_splits/X_train_sm_clean.csv')
X_test = pd.read_csv('../data/processed/train_test_splits/X_test_sm_clean.csv')
y_train = pd.read_csv('../data/processed/train_test_splits/y_train_sm_clean.csv').squeeze()
y_test = pd.read_csv('../data/processed/train_test_splits/y_test_sm_clean.csv').squeeze()

# Drop suspicious features (conservative)
X_train = X_train.drop(columns=['rating_vs_category', 'review_sentiment_avg'], errors='ignore')
X_test = X_test.drop(columns=['rating_vs_category', 'review_sentiment_avg'], errors='ignore')

print(f"📊 Training with {X_train.shape[1]} features")

# ==========================================
# TRAIN XGBOOST
# ==========================================
print("\n🥊 Training XGBoost...")
xgb_model = XGBRegressor(
    n_estimators=1200,
    learning_rate=0.02,
    max_depth=7,
    min_child_weight=3,
    subsample=0.85,
    colsample_bytree=0.85,
    gamma=0,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=42
)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
xgb_r2 = r2_score(y_test, xgb_pred)
print(f"   XGBoost R²: {xgb_r2:.4f}")

# ==========================================
# TRAIN CATBOOST
# ==========================================
print("\n🥊 Training CatBoost...")

cat_model = CatBoostRegressor(
    iterations=1500,
    learning_rate=0.02,
    depth=8,
    l2_leaf_reg=5,
    bagging_temperature=0.5,
    random_strength=0.8,
    border_count=128,
    random_state=42,
    verbose=False
)
cat_model.fit(X_train, y_train)
cat_pred = cat_model.predict(X_test)
cat_r2 = r2_score(y_test, cat_pred)
print(f"   CatBoost R²: {cat_r2:.4f}")

# ==========================================
# FIND BEST BLEND
# ==========================================
print("\n🔍 Testing blend weights...")
best_r2 = 0
best_weight = 0

for w in np.arange(0, 1.1, 0.1):
    blend_pred = w * xgb_pred + (1 - w) * cat_pred
    blend_r2 = r2_score(y_test, blend_pred)
    
    if blend_r2 > best_r2:
        best_r2 = blend_r2
        best_weight = w
    
    print(f"   {w:.1f} XGB + {1-w:.1f} Cat → R²: {blend_r2:.4f}")

print(f"\n🏆 BEST SURVIVAL MODEL:")
print(f"   XGBoost alone:  R² = {xgb_r2:.4f}")
print(f"   CatBoost alone: R² = {cat_r2:.4f}")
print(f"   Blend ({best_weight:.1f}/{1-best_weight:.1f}): R² = {best_r2:.4f}")

# Save both models
joblib.dump(xgb_model, '../models/survival_xgb_v1.pkl')
joblib.dump(cat_model, '../models/survival_cat_v1.pkl')

# Save blend config
blend_config = {
    'xgb_weight': best_weight,
    'cat_weight': 1 - best_weight,
    'r2_score': best_r2
}
import json
with open('../models/survival_blend_config.json', 'w') as f:
    json.dump(blend_config, f)

print(f"\n✅ Saved:")
print(f"   - survival_xgb_v1.pkl")
print(f"   - survival_cat_v1.pkl")
print(f"   - survival_blend_config.json")
print(f"\nBlend weight: {best_weight:.2f} XGB + {1-best_weight:.2f} Cat")


📊 Training with 207 features

🥊 Training XGBoost...
   XGBoost R²: 0.6532

🥊 Training CatBoost...
   CatBoost R²: 0.6354

🔍 Testing blend weights...
   0.0 XGB + 1.0 Cat → R²: 0.6354
   0.1 XGB + 0.9 Cat → R²: 0.6388
   0.2 XGB + 0.8 Cat → R²: 0.6418
   0.3 XGB + 0.7 Cat → R²: 0.6445
   0.4 XGB + 0.6 Cat → R²: 0.6468
   0.5 XGB + 0.5 Cat → R²: 0.6487
   0.6 XGB + 0.4 Cat → R²: 0.6503
   0.7 XGB + 0.3 Cat → R²: 0.6516
   0.8 XGB + 0.2 Cat → R²: 0.6525
   0.9 XGB + 0.1 Cat → R²: 0.6530
   1.0 XGB + 0.0 Cat → R²: 0.6532

🏆 BEST SURVIVAL MODEL:
   XGBoost alone:  R² = 0.6532
   CatBoost alone: R² = 0.6354
   Blend (1.0/0.0): R² = 0.6532

✅ Saved:
   - survival_xgb_v1.pkl
   - survival_cat_v1.pkl
   - survival_blend_config.json

Blend weight: 1.00 XGB + 0.00 Cat


In [17]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score


# --- Base learners (the models you already trained) ---
base_models = [
    ('xgb', xgb_model),
    ('cat', cat_model)
]

# --- Meta learner ---
meta_model = Ridge(alpha=1.0)

# --- Stacking Regressor ---
stack_model = StackingRegressor(
    estimators=base_models,
    final_estimator=meta_model,
    passthrough=False,   # True = include original features too, optional
    cv=5,                # smoothing the meta training
    n_jobs=-1
)

print("\n💞 Training Stacking Regressor...")
stack_model.fit(X_train, y_train)

# Predict
stack_pred = stack_model.predict(X_test)
stack_r2 = r2_score(y_test, stack_pred)

print(f"   Stacking R²: {stack_r2:.4f}")

# Save stack model
joblib.dump(stack_model, '../models/survival_stack_v1.pkl')

print("\n💘 Saved: survival_stack_v1.pkl")
print(f"🏆 Stacking Model R²: {stack_r2:.4f}")



💞 Training Stacking Regressor...
   Stacking R²: 0.6495

💘 Saved: survival_stack_v1.pkl
🏆 Stacking Model R²: 0.6495


In [18]:
joblib.dump(stack_model, '../models/survival_month_model/sm_v1/survival_month_best_stack_model.pkl')
joblib.dump(xgb_model, '../models/survival_month_model/sm_v1/survival_month_xgb_model.pkl')
joblib.dump(cat_model, '../models/survival_month_model/sm_v1/survival_month_cat_model.pkl')
joblib.dump(X_train, '../models/survival_month_model/sm_v1/feature_names.pkl')
print("Saved!")

Saved!


In [19]:
X_train.shape

(35674, 207)